# Fine-tuning LoRA CimSuggester - Sovereign OS DIM (a lancer sur GPU)

Entraine un adaptateur LoRA reel pour le suggesteur CIM-10 (`Qwen/Qwen2.5-0.5B-Instruct`), a partir du dataset synthetique de `backend/ml/gen_cim_lora_dataset.py`.

**Pourquoi ce notebook** : un run complet (`max_steps=600`) prend environ 30h en CPU pur (mesure reelle : ~178 s/step sur une machine sans GPU). Sur un GPU T4 gratuit (Colab), la meme charge de travail se termine en quelques minutes a quelques dizaines de minutes.

**Avant de lancer** : `Execution > Modifier le type d'execution > Accelerateur materiel > GPU (T4)`.

**A la fin** : la derniere cellule telecharge un zip contenant `cim_lora_adapter/` + `cim_lora_training_meta.json`. Decompresser dans `backend/ml/models/` du depot local, puis commiter normalement (meme convention que les autres artefacts ML du repo).

In [ ]:
import torch
assert torch.cuda.is_available(), "Aucun GPU detecte - Execution > Modifier le type d'execution > GPU"
print("GPU :", torch.cuda.get_device_name(0))

In [ ]:
# Clone du depot (public, pas d'auth necessaire). Branche feat/cim-lora-finetune
# contient deja le dataset genere + le script d'entrainement.
!git clone --branch feat/cim-lora-finetune --depth 1 https://github.com/Adam-Blf/sovereign_os_dim.git
%cd sovereign_os_dim

In [ ]:
# Colab fournit deja torch avec CUDA - on installe uniquement les extras
# LoRA (peft/datasets/accelerate), pas torch/transformers en entier pour
# eviter d'ecraser la version CUDA deja en place.
!pip install -q peft>=0.19.0 accelerate>=1.14.0 "datasets>=5.0.0"

In [ ]:
# Dataset deja genere et commite (backend/ml/data/cim_lora_dataset.jsonl,
# seed=42, deterministe) - regenerer n'est utile que si le generateur a
# change depuis le dernier commit clone.
!python -m backend.ml.gen_cim_lora_dataset

In [ ]:
# Run complet - detecte automatiquement le GPU (bf16, batch plus large).
# max_steps reste le cap dur (pas epoch-based), meme garde-fou qu'en CPU.
!python -m backend.ml.train_cim_lora --max-steps 600

In [ ]:
# Verification : le fichier meta ne s'ecrit qu'apres sauvegarde reussie de
# l'adaptateur - son absence = echec de l'entrainement.
import json
from pathlib import Path
meta_path = Path("backend/ml/models/cim_lora_training_meta.json")
assert meta_path.exists(), "Entrainement incomplet - cim_lora_training_meta.json absent"
print(json.dumps(json.loads(meta_path.read_text(encoding="utf-8")), indent=2, ensure_ascii=False))

In [ ]:
# Zippe l'adaptateur + les metadonnees et propose le telechargement.
from google.colab import files

!zip -rq cim_lora_adapter_output.zip backend/ml/models/cim_lora_adapter backend/ml/models/cim_lora_training_meta.json
files.download("cim_lora_adapter_output.zip")

## Apres telechargement

1. Decompresser `cim_lora_adapter_output.zip` a la racine du depot local (`sovereign_os_dim/`) - il reconstitue `backend/ml/models/cim_lora_adapter/` et `backend/ml/models/cim_lora_training_meta.json`.
2. Verifier `cim_lora_training_meta.json` (perte finale, steps completes, duree).
3. Suivre la suite du plan Flux B : conversion GGUF via `llama.cpp` (`convert_lora_to_gguf.py`), mise a jour de `tools/ollama/Modelfile.sovereign-cim`, controle qualitatif manuel avant de promouvoir `sovereign-cim-lora`.